# Sarcasm-Aware Sentiment Ensemble

This notebook documents the sentiment-analysis methodology developed for **Meridian**, a career
intelligence platform for graduate students. One component scores open-text student feedback, where
standard sentiment tools fail in a predictable way: they misread sarcasm, and sarcasm is common in
candid feedback.

The approach combines two complementary sentiment models, layers a high-precision sarcasm-detection
step on top, and treats a sarcasm cue as strong evidence that surface positivity should be inverted.
The production system runs this over tens of thousands of records; here it runs on a small set of
**synthetic** comments so the method is fully reproducible without any institutional data.

*Author: Abby Williams, PhD. Synthetic data only; no student data is included. The cue list here is a
representative subset of the larger, feedback-tuned list used in production.*

## The problem

Two widely used lexicon-based tools have complementary strengths:

- **TextBlob** is a serviceable general-purpose polarity scorer but literal. It reads "Wonderful, another
  7am session" as positive because it keys on "wonderful."
- **VADER** is tuned for informal text and handles intensity and punctuation well, but on its own it still
  misses sarcasm that depends on context rather than tokens.

In candid feedback, sarcasm is a recurring failure mode, and it is exactly the negative signal an advising
system most needs to surface. A naive average of the two models inherits both blind spots. The design here
makes a deliberate, inspectable tradeoff: detect likely sarcasm from known cues, and when a cue is present,
treat the surface sentiment as ironic and pull the score negative in proportion to how positive the surface
looks. Strong positive language plus a sarcasm cue is read as strong negativity.

In [1]:
import re
import contractions
import pandas as pd
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

## Step 1: Preprocessing

Text is normalized in ways that matter for informal feedback: contractions are expanded so "can't" reads
consistently, repeated characters are collapsed ("sooooo" to "soo") so emphasis does not distort matching,
and runs of punctuation are reduced while "!" and "?" are kept because they carry signal.

In [2]:
def preprocess_text_for_sentiment(text):
    """Normalize informal text for sentiment scoring."""
    if not isinstance(text, str):
        return ""
    text = contractions.fix(text)               # can't -> cannot
    text = text.lower()
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)    # sooooo -> soo
    text = re.sub(r"([!?.])\1+", r"\1", text)      # collapse repeated punctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text

preprocess_text_for_sentiment("I can't WAIT for another meeting!!!")

'i cannot wait for another meeting!'

## Step 2: Sarcasm cues

Each cue carries a negative valence. Detection is a substring check; scoring uses the valence directly,
which sidesteps a subtle trap: VADER does word-level lookups, so multi-word phrases like "thanks for
nothing" never fire if you only inject them into VADER's lexicon. Scoring the cues directly is what makes
the multi-word markers actually count.

The cue list is deliberately domain-aware. Phrases like "good luck with that" and "thanks for nothing"
recur in candid feedback and almost always signal frustration despite containing no overtly negative
words.

In [3]:
SARCASM_LEXICON = {
    # multi-word cues (scored directly, since VADER would not catch them)
    "yeah right": -2.0, "as if": -2.0, "thanks for nothing": -2.0, "good luck with that": -2.0,
    "what a surprise": -1.5, "what a joy": -1.5, "thanks a lot": -1.5, "sure thing": -1.0,
    "love that": -1.5, "just perfect": -2.0, "so helpful": -1.5, "i bet": -1.0, "/s": -2.0,
    # single-word cues
    "wonderful": -1.5, "fantastic": -1.5, "brilliant": -1.5, "amazing": -1.0,
    "lovely": -1.0, "obviously": -1.0,
}

def find_cues(text):
    """Return any sarcasm cues present in the (preprocessed) text."""
    return [cue for cue in SARCASM_LEXICON if cue in text]

## Step 3: The ensemble

For a comment with no cue, the score is a weighted blend of VADER and TextBlob. For a flagged comment,
two pieces combine: the average valence of the cues present, and the inverted surface sentiment (the more
positive the surface, the more negative the reading, because strong positive language under a sarcasm cue
is the signature of irony). The blend guarantees a flagged comment does not return a positive score.

In [4]:
def calculate_sentiment(text):
    """Return per-model and ensemble sentiment, with sarcasm-aware correction."""
    pre = preprocess_text_for_sentiment(text)
    cues = find_cues(pre)
    sarcasm = len(cues) > 0

    tb_score = TextBlob(pre).sentiment.polarity
    vader_score = SentimentIntensityAnalyzer().polarity_scores(pre)["compound"]
    surface = 0.6 * vader_score + 0.4 * tb_score

    if not sarcasm:
        ensemble = surface
    else:
        cue_valence = sum(SARCASM_LEXICON[c] for c in cues) / len(cues) / 2.0  # normalize toward [-1, 0]
        inverted_surface = -abs(surface)                                       # irony: positive -> negative
        ensemble = 0.6 * cue_valence + 0.4 * inverted_surface

    ensemble = max(-1.0, min(1.0, ensemble))
    return {
        "TextBlob": round(tb_score, 3),
        "VADER": round(vader_score, 3),
        "Ensemble": round(ensemble, 3),
        "Sarcasm": sarcasm,
    }

calculate_sentiment("Wonderful, the session was scheduled at 7am again.")

{'TextBlob': 1.0, 'VADER': 0.572, 'Ensemble': -0.747, 'Sarcasm': True}

## Demonstration on synthetic feedback

The comments below are invented for illustration and span sincere positive, sincere negative, neutral, and
sarcastic cases. The final row is included deliberately as a case the method misses, discussed in the
limitations.

In [5]:
synthetic_comments = [
    "The career workshop genuinely helped me clarify my postdoc options.",
    "I could not get any useful feedback on my industry applications.",
    "I attended the session on Tuesday afternoon.",
    "Yeah right, like the academic job market is totally fine.",
    "Thanks for nothing, the resume review was canceled twice.",
    "Wonderful, the session was scheduled at 7am again.",
    "Good luck with that, the funding portal has been broken for weeks.",
    "Oh great, another networking event during finals week.",  # no listed cue -> missed
]

rows = []
for c in synthetic_comments:
    s = calculate_sentiment(c)
    label = "negative" if s["Ensemble"] <= -0.05 else "positive" if s["Ensemble"] >= 0.05 else "neutral"
    rows.append({
        "Comment": c,
        "TextBlob": s["TextBlob"],
        "VADER": s["VADER"],
        "Ensemble": s["Ensemble"],
        "Reading": label,
        "Sarcasm": "yes" if s["Sarcasm"] else "",
    })

results_df = pd.DataFrame(rows)
results_df

,Comment,TextBlob,VADER,Ensemble,Reading,Sarcasm
0,The career workshop genuinely helped me clarif...,0.400,0.000,0.160,positive,
1,I could not get any useful feedback on my indu...,0.300,-0.341,-0.085,negative,
2,I attended the session on Tuesday afternoon.,0.000,0.000,0.000,neutral,
3,"Yeah right, like the academic job market is to...",0.234,0.700,-0.805,negative,yes
4,"Thanks for nothing, the resume review was canc...",0.200,0.440,-0.738,negative,yes
5,"Wonderful, the session was scheduled at 7am ag...",1.000,0.572,-0.747,negative,yes
6,"Good luck with that, the funding portal has be...",0.150,0.421,-0.725,negative,yes
7,"Oh great, another networking event during fina...",0.800,0.625,0.695,positive,


## Where the ensemble earns its keep

The cued sarcastic rows are the point. A literal baseline (TextBlob alone) reads them as positive; the
ensemble pulls them negative, which is the correct reading for an advising signal.

In [6]:
cued = results_df[results_df["Sarcasm"] == "yes"][["Comment", "TextBlob", "Ensemble"]].copy()
cued = cued.rename(columns={"TextBlob": "TextBlob alone (baseline)", "Ensemble": "Ensemble (final)"})
cued.reset_index(drop=True)

,Comment,TextBlob alone (baseline),Ensemble (final)
0,"Yeah right, like the academic job market is to...",0.234,-0.805
1,"Thanks for nothing, the resume review was canc...",0.200,-0.738
2,"Wonderful, the session was scheduled at 7am ag...",1.000,-0.747
3,"Good luck with that, the funding portal has be...",0.150,-0.725


## Limitations

This is a deliberate engineering tradeoff, not a state-of-the-art sarcasm classifier, and it is worth being
precise about what it does and does not do.

Cue-based detection is **high precision, lower recall**. When a known cue is present, the negative reading
is reliable. But sarcasm that uses no listed cue is missed and falls through to the base ensemble. The final
row above is exactly this case: "Oh great, another networking event" carries no cue in the list, so it is
scored positive. Catching it would mean either expanding the cue list (raising false-positive risk on
sincere uses of "great") or moving to a learned classifier.

That tradeoff was intentional. For an advisor-facing tool, an inspectable rule layer that a non-technical
user can trust and audit was preferred over a black-box classifier, and the asymmetric cost of errors made
precision on flagged negatives the priority. A learned model would raise recall at the cost of
interpretability and speed; in production, the cue list was tuned against the real feedback distribution,
which lifted recall well above what an adversarial example set like this one suggests.

The honest summary: the method reliably catches cued sarcasm and is transparent about the rest. In the full
system, this scoring feeds a thematic layer that aggregates sentiment by topic and academic division for
advisors. The production code is private due to institutional data agreements; this notebook isolates the
sentiment methodology so the approach is legible on its own.